# Глава 5. Ретриверы и поиск информации

## 5.1. Типы ретриверов и их особенности

Ретривер — это главная часть RAG-системы. Он решает, какие данные получает языковая модель для ответа. Сегодня есть много разных ретриверов: от простых (поиск по словам) до сложных (понимают не только текст, но и картинки).

Выбор ретривера сильно влияет на качество работы всей системы. Исследования показали: если ретривер подобран плохо, результат ухудшается на 20–40%. Даже если языковая модель очень хорошая.

Векторные ретриверы: поиск по смыслу

Векторные ретриверы — основа современных RAG-систем. Они понимают смысл текста. Как они работают: превращают документы и запросы в числа (векторы). Похожие по смыслу тексты оказываются рядом в пространстве этих чисел.

Простой векторный ретривер ищет самые близкие по смыслу документы. Сделать такой ретривер очень легко:

In [ ]:
# Этот код создаёт компонент для поиска по векторной базе данных, который будет возвращать **10 самых похожих документов** на заданный запрос.
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

Система сама превращает запросы в числа (эмбеддинги) и сравнивает их с числами документов, которые посчитала заранее.

Чтобы не выдавать неподходящие результаты, добавляют порог схожести. Настройка `similarity_score_threshold` помогает отсеять документы с низкой схожестью.

In [ ]:
# Этот код настраивает поиск так, чтобы он возвращал **только те документы, чья степень сходства с запросом выше 0.5** (то есть отсекает недостаточно релевантные результаты).
search_type = "similarity_score_threshold",
search_kwargs={"score_threshold": 0.5}

Это особенно важно для запросов по узкой теме.

Dense Passage Retriever (DPR) — это классический инструмент от Facebook AI. Систему обучают на парах «вопрос — документ». Она учится делать числа (эмбеддинги) так, чтобы хорошо отличать подходящие отрывки от неподходящих. Модель состоит из двух частей: одна обрабатывает вопросы, другая — документы.

### Лексические ретриверы: точность терминологических совпадений

Алгоритм BM25 — это лучший способ искать по словам в RAG-системах. Он считает, насколько документ подходит под запрос. Для этого он смотрит:

- как часто слово встречается в документе;
- насколько это слово редкое во всей коллекции;
- длину документа (чтобы длинные тексты не имели преимущества).

BM25Retriever в LangChain создаётся из документов простой командой:

In [ ]:
sparse_retriever = BM25Retriever.from_documents(documents, k=10)

Система сама строит обратный индекс и считает статистику по словам для быстрого поиска.

Плюсы BM25:
- работает очень быстро;
- отлично находит точные совпадения по словам;
- не нужно заранее обучать модель;
- работает с любыми языками;
- особенно хорош для технических документов, FAQ и юридических текстов.

TF‑IDF — это похожая модель, но с более простой формулой. Она тоже считает важность слов, но не учитывает длину документа.

### Гибридные ретриверы: синтез подходов

Ensemble Retriever объединяет плюсы разных методов поиска. Он собирает результаты от нескольких ретриверов и смешивает их с определёнными весами. Обычно используют вместе плотный и разреженный ретривер с одинаковыми весами:

In [ ]:
# Этот код создаёт гибридный поисковик, который объединяет результаты плотного (смыслового) и разреженного (по ключевым словам) поиска,
# усредняя их оценки с равным весом 0.5 на каждого для финального ранжирования.
EnsembleRetriever(retrievers=[dense_retriever, sparse_retriever], weights=[0.5, 0.5])

Reciprocal Rank Fusion (RRF) помогает хорошо объединять результаты от разных ретриверов. Алгоритм сортирует документы по их местам в каждом отдельном списке результатов. Это позволяет честно сравнивать разные оценки релевантности.

Гибридные системы (где объединены несколько методов) почти всегда работают лучше, чем системы с одним методом. Векторный поиск хорошо понимает смысл, а лексический — точно находит слова. Вместе они надёжно обрабатывают запросы разного типа.

### Кросс‑энкодерные ретриверы: глубокое понимание

Кросс-энкодеры смотрят на пару «запрос — документ» вместе, одновременно. Это позволяет модели улавливать сложные связи между словами. В отличие от би-энкодеров (которые обрабатывают запрос и документ по отдельности), кросс-энкодер подаёт в модель склеенный текст.

Плюс кросс-энкодеров — они отлично ранжируют результаты для сложных запросов, где важен контекст и мелкие детали. Модели на основе BERT или RoBERTa показывают лучшие результаты в вопросно-ответных системах.

Главный минус — медленная работа. Каждую пару «запрос — документ» модель прогоняет заново. Поэтому кросс-энкодер нельзя использовать для быстрого поиска по большой коллекции. Обычно его применяют так: другой ретривер находит несколько лучших документов, а кросс-энкодер уже переранжирует только их.

### MMR: баланс релевантности и разнообразия

Maximal Marginal Relevance (MMR) решает проблему повторов в результатах поиска. Алгоритм находит баланс между тем, насколько документ подходит под запрос, и тем, насколько он отличается от уже отобранных. Это помогает не выдавать кучу похожих документов.

Формула MMR устроена так: система считает схожесть документа с запросом и вычитает из неё максимальную схожесть с уже выбранными документами.

`λ⋅similarity(query,doc)−(1−λ)⋅max_similarity(doc,selected_docs)`

Параметр `λ` задаёт баланс: что важнее — релевантность или разнообразие. Обычно его ставят в пределах 0,5–0,7.

---

Эта формула — классический пример **гибридного ранжирования (Hybrid Scoring)**, часто используемого в задачах поиска информации (Information Retrieval), например, в системах вроде RAG (Retrieval-Augmented Generation) для борьбы с дубликатами.

Она позволяет соблюсти баланс между релевантностью документа запросу и его новизной по отношению к уже отобранным документам.

### Общий вид
$$Score = \lambda \cdot S_{rel} - (1-\lambda) \cdot S_{div}$$

Где:
*   $S_{rel}$ = `similarity(query, doc)` — релевантность запросу.
*   $S_{div}$ = `max_similarity(doc, selected_docs)` — максимальное сходство с уже выбранным.
*   $\lambda$ — коэффициент баланса (от 0 до 1).

### Разбор действий по шагам

#### 1. `similarity(query, doc)` — Левая часть (Сила притяжения)
**Что это:** Это косинусное сходство (или другое расстояние) между вектором вопроса пользователя и вектором документа-кандидата.
**Почему так:** Это основа основ. Если пользователь спросил «Как приготовить борщ», документ про рецепт борща получает высокий балл. Эта часть тянет результат вверх. Чем выше сходство, тем лучше.

#### 2. `max_similarity(doc, selected_docs)` — Правая часть (Сила отталкивания)
**Что это:** Мы берем документ-кандидат и сравниваем его со *всеми* документами, которые мы уже отобрали в финальный список на предыдущих шагах (жадный алгоритм). Мы выбираем самое большое значение сходства.
**Почему ищем максимум (max):**
Нам не важно, насколько кандидат похож на все старые документы в среднем. Нам важно, нет ли у него *близнеца*. Если он очень похож хотя бы на один из уже выбранных документов (`max = 0.99`), значит, он — дубликат. Наказание должно быть максимальным. Если максимум маленький (`max = 0.2`), значит, документ несет новую информацию.

#### 3. `1 - λ` — Вес штрафа за дублирование
**Почему так:** Если λ (лямбда) — это вес релевантности, то `1 - λ` — это то, что осталось от единицы. Это математический способ сказать: «Баланс ручки настройки». Если λ = 0.7, то штраф будет 0.3. Сумма весов всегда равна 1. Это гарантирует, что мы не выйдем за пределы разумных значений скоров.

#### 4. Знак минус: `(Лямбда) - (1-Лямбда)`
**Почему вычитаем:** Потому что мы хотим **наказать** за сходство с уже выбранным.
*   Левая часть дает плюс за релевантность.
*   Правая часть (сходство с отобранными) — это всегда положительное число. Если мы будем его прибавлять, мы будем поощрять копии.
*   Поставив знак минус, мы говорим: «Чем ты более похож на уже существующий ответ, тем хуже твой итоговый балл».

### Зачем это нужно? Моделирование ситуации MMR
Эта формула — упрощенная версия алгоритма **MMR (Maximal Marginal Relevance)**.

Представьте, что пользователь спросил: *«Документы про яблоки»*.
В базе есть 3 документа:
1.  *«Яблоки — это фрукты»* (Релевантность: 0.9)
2.  *«Яблоки — это фрукты (копия)»* (Релевантность: 0.9)
3.  *«Пирог с яблоками»* (Релевантность: 0.7)

**Если использовать только левую часть (λ=1):**
Ранжирование: Док 1 (0.9), Док 2 (0.9), Док 3 (0.7).
Результат: Пользователь получит два одинаковых ответа про фрукты и не увидит пирог. Ответ глупый и неполный.

**Используем формулу (λ=0.7):**

*   *Шаг 1 (Выбираем Док 1):* Список `selected_docs` пуст. Штрафа нет. Score = 0.7 * 0.9 - 0 = **0.63**. Выбираем.
*   *Шаг 2 (Кандидаты Док 2 и Док 3):*
    *   Док 2: Релевантность 0.9. Сходство с Доком 1 = 0.99.
        Score = 0.7 * 0.9 – 0.3 * 0.99 = 0.63 – 0.297 = **0.33** (Упал).
    *   Док 3: Релевантность 0.7. Сходство с Доком 1 = 0.1.
        Score = 0.7 * 0.7 – 0.3 * 0.1 = 0.49 – 0.03 = **0.46** (Выше!).

Результат: Мы выбрали Док 3 (пирог) вместо копии Дока 1.

### Резюме
Формула заставляет алгоритм искать компромисс: «Найди документ, который **похож на запрос** (вопрос пользователя), но при этом **не похож на то, что ты уже показал** (разнообразие)».

---

Использовать MMR в LangChain очень просто: нужно указать `search_type="mmr"` при создании ретривера. Система сама применит алгоритм при векторном поиске. Это заметно улучшает результаты для общих, обзорных запросов.

### Специализированные ретриверы

Контекстные ретриверы помнят историю разговора и предпочтения пользователя. Они следят за тем, как идёт диалог, и подстраивают поиск под текущий контекст.

Графовые ретриверы работают с базами знаний, где объекты связаны между собой. RAG с графом знаний понимает связи между понятиями. Это особенно важно для сложных аналитических запросов.

Мультимодальные ретриверы работают с текстом, картинками и звуком в единой системе. Модели, похожие на CLIP, умеют искать картинки по тексту и наоборот.